# Sustainability Aware Asset Management
## Group BB — Emerging Markets | Scope 1

---

**Matthieu Pagot, Aymen Laraoui, Cédric Nim, Rijad Talovic**

---
## Libraries & Configuration

In [28]:
# Libraries import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from joblib import Parallel, delayed

In [29]:
#Data path
data_path = 'Data_2026/' #changer si nécessaire

#Group Parameters
region = "EM"

#Configuration
low_prices_threshold = 0.5
stale_threshold = 0.5 # Proportion max de rendements nuls tolérée (50%)

---
# Part I — Standard Portfolio Allocation

## 1. Data Handling

### 1.1 Data Loading

In [30]:
#Static file Loading and filtering to emerging markets

static_file = pd.read_excel(f'{data_path}Static_2025.xlsx')
emerging_market_data = static_file[static_file['Region'] == region]
#print(emerging_market_data)

#Function to load Data files
def load_data(file_name):
  data = pd.read_excel(f'{data_path}{file_name}')
  data = data.dropna(subset=['ISIN']) # drop error rows
  data = data[data['ISIN'].isin(emerging_market_data['ISIN'])] #Keep emerging markets only
  data = data.set_index('ISIN') #Make ISIN the index
  data = data.drop(columns=['NAME']) #Drop the name column
  data = data.loc[:, data.columns.notna()] #drop columns with NaN
  data = data.apply(pd.to_numeric, errors='coerce') #Tries to convert every cell to a number and if it can't turn it into NaN

  return data

In [31]:
#Load all the data files

#Return Index
ri_monthly = load_data('DS_RI_T_USD_M_2025.xlsx') # monthly total return index (prices)
ri_yearly  = load_data('DS_RI_T_USD_Y_2025.xlsx') # yearly total return index

#Market Capitalization
mv_monthly = load_data('DS_MV_T_USD_M_2025.xlsx') # monthly market capitalization
mv_yearly  = load_data('DS_MV_T_USD_Y_2025.xlsx') # yearly market capitalization

#Co2 Scope 1 Emissions
co2_scope1 = load_data('DS_CO2_SCOPE_1_Y_2025.xlsx') # yearly CO2 scope 1 emissions (tonnes)
co2_scope1 = co2_scope1.ffill(axis=1) # forward-fill les années manquantes (consigne PDF section 1)

#Revenues
revenues   = load_data('DS_REV_Y_2025.xlsx') # yearly revenues (thousands USD)
revenues   = revenues.ffill(axis=1) # forward-fill les années manquantes (consigne PDF section 1)

#petit aperçu de la data
print(static_file.columns.tolist())
print(emerging_market_data[['NAME', 'Country', 'Region']].head())

['ISIN', 'NAME', 'Country', 'Region']
                                   NAME Country Region
1                                 ALUAR      AR     EM
2                  BANCO BBVA ARGENTINA      AR     EM
3    TERNIUM ARGENTINA SOCIEDAD ANONIMA      AR     EM
117              CHINA YURUN FOOD GROUP      CN     EM
144                            AMBEV ON      BR     EM


In [32]:
#Load the Risk Free Rate

rf_raw = pd.read_excel(f'{data_path}Risk_Free_Rate_2025.xlsx')
rf_raw.columns = ['Date', 'Risk Free Rate']
rf_raw['Date'] = pd.to_datetime(rf_raw['Date'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
rf = rf_raw.set_index('Date')['Risk Free Rate'] / 100

### 1.2 Data Cleaning

In [33]:
#Remove the firms with no data

# Prix bas : Remplacer les valeurs strictement inférieures au seuil par NaN
ri_monthly[ri_monthly < low_prices_threshold] = np.nan

# Boucher les trous : Combler les valeurs manquantes isolées avec le prix précédent
ri_monthly = ri_monthly.ffill(limit=1, axis=1)

In [34]:
# Calculer les rendements mensuels à partir des prix
prices = ri_monthly.copy()

# Calculer les rendements simples (pct_change sur les colonnes = dans le temps)
returns = prices.pct_change(axis=1)

# Gérer les faillites (delistings) : forcer le rendement à -100%, le masque repère : mois t-1 avait un prix ET mois t est vide
mask_delisted = prices.notna().shift(1, axis=1) & prices.isna()
returns[mask_delisted] = -1.0

# Nettoyer les infinis (divisions par zéro)
returns = returns.replace([np.inf, -np.inf], np.nan)

print(f"Shape des returns : {returns.shape}")
print(f"Valeurs infinies restantes : {np.isinf(returns.values).sum()}")
print(returns.head())

Shape des returns : (702, 314)
Valeurs infinies restantes : 0
              1999-12-31 00:00:00  2000-01-31 00:00:00  2000-02-29 00:00:00  \
ISIN                                                                          
ARALUA010258                  NaN             0.176511             0.050002   
ARP125991090                  NaN            -0.051861             0.178923   
ARSIDE010029                  NaN            -0.004877             0.039543   
BMG211591018                  NaN                  NaN                  NaN   
BRABEVACNOR1                  NaN            -0.044175            -0.045539   

              2000-03-31 00:00:00  2000-04-28 00:00:00  2000-05-31 00:00:00  \
ISIN                                                                          
ARALUA010258            -0.023552            -0.016173            -0.016727   
ARP125991090            -0.105072            -0.161842             0.005304   
ARSIDE010029            -0.002113            -0.071188            -0

---
## 2. Minimum-Variance Portfolio Allocation

### 2.1 Investment Set

Pour chaque année d'investissement Y (de 2014 à 2024), on constitue l'univers investissable à partir des données disponibles fin Y-1. Les variables construites sont :

| Variable | Description |
|---|---|
| `emerging_market_data` | Entreprises EM (filtre région) |
| `eligible_isins` | ISINs EM ayant déclaré leurs émissions CO2 fin Y-1 |
| `filtered_isins` | Survivants aux 3 filtres financiers (historique, stale, dernier prix) |
| `filtered_carbon_isin_dict` | Dictionnaire final `{année: [ISINs éligibles]}` pour l'optimisation |

In [35]:
# Pour avoir le set d'investissment possible par année, on doit appliquer les filtres suivants :

filtered_carbon_isin_dict = {}

for year in range(2014, 2026): # 2014 à 2025 inclus = 12 ans = 144 mois (T=144 requis par le PDF)

    # Fenêtre de temps dynamique de 10 ans
    end_date = returns.columns[returns.columns.astype(str).str.contains(str(year-1))].max()
    end_idx = returns.columns.get_loc(end_date)
    start_idx = end_idx - 119
    window_cols = returns.columns[start_idx:end_idx+1]

    # ÉTAPE 2 : FILTRE CARBONE — ISINs ayant déclaré des émissions CO2 pour Y-1
    prev_year = year - 1
    matching_cols = [col for col in co2_scope1.columns
                     if str(col) == str(prev_year) or (hasattr(col, 'year') and col.year == prev_year)]
    eligible_isins = co2_scope1[matching_cols[0]].dropna().index.tolist() if matching_cols else []

    filtered_isins = []

    # ÉTAPE 3 : FILTRES FINANCIERS
    for isin in eligible_isins:
        if isin not in returns.index:
            continue
        returns_series = returns.loc[isin, window_cols]

        # FILTRE A : min 36 mois valides sur 120
        has_enough_data = returns_series.notna().sum() >= 36

        # FILTRE B : exclure si proportion de rendements nuls > stale_threshold
        has_stale_prices = (returns_series == 0.0).sum() / 120 > stale_threshold

        # FILTRE C : dernier PRIX valide (et non dernier rendement — cf. consigne PDF)
        last_price_valid = (isin in prices.index) and pd.notna(prices.loc[isin, end_date])

        if has_enough_data and not has_stale_prices and last_price_valid:
            filtered_isins.append(isin)

    filtered_carbon_isin_dict[year] = filtered_isins

    print(f"Annee {year} (Fenetre du {window_cols[0].date()} au {window_cols[-1].date()}) :")
    print(f"  -> {len(filtered_isins)} actions conservees sur {len(eligible_isins)} eligibles au depart.")

Annee 2014 (Fenetre du 2004-01-30 au 2013-12-31) :
  -> 254 actions conservees sur 286 eligibles au depart.
Annee 2015 (Fenetre du 2005-01-31 au 2014-12-31) :
  -> 280 actions conservees sur 313 eligibles au depart.
Annee 2016 (Fenetre du 2006-01-31 au 2015-12-31) :
  -> 312 actions conservees sur 349 eligibles au depart.
Annee 2017 (Fenetre du 2007-01-31 au 2016-12-30) :
  -> 346 actions conservees sur 383 eligibles au depart.
Annee 2018 (Fenetre du 2008-01-31 au 2017-12-29) :
  -> 389 actions conservees sur 428 eligibles au depart.
Annee 2019 (Fenetre du 2009-01-30 au 2018-12-31) :
  -> 425 actions conservees sur 467 eligibles au depart.
Annee 2020 (Fenetre du 2010-01-29 au 2019-12-31) :
  -> 472 actions conservees sur 521 eligibles au depart.
Annee 2021 (Fenetre du 2011-01-31 au 2020-12-31) :
  -> 512 actions conservees sur 565 eligibles au depart.
Annee 2022 (Fenetre du 2012-01-31 au 2021-12-31) :
  -> 547 actions conservees sur 603 eligibles au depart.
Annee 2023 (Fenetre du 2013-

In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# FONCTION : covariance pairwise (Common Sample — Lecture 5 slides 20-21)
# ─────────────────────────────────────────────────────────────────────────────

def compute_pairwise_cov(i, j, ret_matrix, first_valid, last_valid, variances, means):
    """
    Calcule la covariance entre les actifs i et j selon la méthode Common Sample.

    Étape A : moyennes globales pré-calculées sur l'historique valide complet (10 ans)
    Étape B : variances pré-calculées sur l'historique valide complet (ddof=1)
    Étape C : fenêtre d'overlap = [max(first_i, first_j), min(last_i, last_j)]
    Étape D : corrélation sur l'overlap avec la MOYENNE GLOBALE (formule exacte Cours 5)
              → on NE PAS utiliser np.corrcoef (utilise la moyenne locale de l'overlap)
    Étape E : Cov(i,j) = rho * sqrt(Var_i) * sqrt(Var_j)
    """
    var_i = variances[i]
    var_j = variances[j]

    # Variance nulle ou manquante → covariance nulle
    if var_i <= 0 or var_j <= 0 or np.isnan(var_i) or np.isnan(var_j):
        return i, j, 0.0

    # Étape C : bornes de l'overlap
    tau1 = max(first_valid[i], first_valid[j])
    taun = min(last_valid[i],  last_valid[j])

    # Pas d'overlap suffisant → covariance nulle
    if tau1 >= taun:
        return i, j, 0.0

    # Extraction de la fenêtre d'overlap
    r_i_ov = ret_matrix[i, tau1:taun + 1]
    r_j_ov = ret_matrix[j, tau1:taun + 1]

    # Étape D : garder uniquement les mois où les deux séries sont valides
    valid_mask = ~np.isnan(r_i_ov) & ~np.isnan(r_j_ov)
    r_i_clean  = r_i_ov[valid_mask]
    r_j_clean  = r_j_ov[valid_mask]

    if len(r_i_clean) < 2:
        return i, j, 0.0

    # Corrélation avec la MOYENNE GLOBALE (formule Cours 5 — pas np.corrcoef)
    # R̄_i = moyenne sur l'historique complet (10 ans), pas sur l'overlap seul
    num   = np.sum((r_i_clean - means[i]) * (r_j_clean - means[j]))
    den_i = np.sqrt(np.sum((r_i_clean - means[i]) ** 2))
    den_j = np.sqrt(np.sum((r_j_clean - means[j]) ** 2))

    if den_i == 0 or den_j == 0:
        return i, j, 0.0

    corr = num / (den_i * den_j)

    if np.isnan(corr):
        return i, j, 0.0

    # Étape E : reconstruction de la covariance
    cov = corr * np.sqrt(var_i) * np.sqrt(var_j)

    return i, j, cov


# ─────────────────────────────────────────────────────────────────────────────
# CALCUL DES MATRICES DE COVARIANCE — une par année d'investissement
# ─────────────────────────────────────────────────────────────────────────────

covariance_matrices = {}

for year in range(2014, 2026):

    # Reconstruction de la fenêtre de 120 mois (identique à la section 2.1)
    end_date    = returns.columns[returns.columns.astype(str).str.contains(str(year - 1))].max()
    end_idx     = returns.columns.get_loc(end_date)
    start_idx   = end_idx - 119
    window_cols = returns.columns[start_idx:end_idx + 1]

    isins = filtered_carbon_isin_dict[year]
    N     = len(isins)

    # Matrice de rendements (N x 120) en numpy pour performance
    ret_matrix = returns.loc[isins, window_cols].values  # shape (N, 120)

    # ── Étapes A & B : pré-calcul des moyennes globales, variances et bornes ──
    first_valid = np.full(N, 120, dtype=int)  # valeur par défaut : hors fenêtre
    last_valid  = np.full(N, -1,  dtype=int)
    variances   = np.zeros(N)
    means       = np.zeros(N)  # Étape A : moyenne globale sur l'historique valide

    for i in range(N):
        r         = ret_matrix[i]
        valid_idx = np.where(~np.isnan(r))[0]
        if len(valid_idx) >= 2:
            first_valid[i] = valid_idx[0]
            last_valid[i]  = valid_idx[-1]
            means[i]       = np.mean(r[valid_idx])         # Étape A : moyenne globale
            variances[i]   = np.var(r[valid_idx], ddof=1)  # Étape B : variance (non-biaisée)

    # ── Initialisation de la matrice : variances sur la diagonale ─────────────
    sigma = np.zeros((N, N))
    np.fill_diagonal(sigma, variances)

    # ── Triangle supérieur en parallèle ───────────────────────────────────────
    pairs = list(itertools.combinations(range(N), 2))

    results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(compute_pairwise_cov)(i, j, ret_matrix, first_valid, last_valid, variances, means)
        for i, j in pairs
    )

    # ── Remplissage symétrique ─────────────────────────────────────────────────
    for i, j, cov in results:
        sigma[i, j] = cov
        sigma[j, i] = cov

    covariance_matrices[year] = sigma

    print(f"Annee {year} : matrice {N}x{N} calculee "
          f"({len(pairs):,} paires, fenetre {window_cols[0].date()} -> {window_cols[-1].date()})")

Annee 2014 : matrice 254x254 calculee (32,131 paires, fenetre 2004-01-30 -> 2013-12-31)
Annee 2015 : matrice 280x280 calculee (39,060 paires, fenetre 2005-01-31 -> 2014-12-31)
Annee 2016 : matrice 312x312 calculee (48,516 paires, fenetre 2006-01-31 -> 2015-12-31)
Annee 2017 : matrice 346x346 calculee (59,685 paires, fenetre 2007-01-31 -> 2016-12-30)
Annee 2018 : matrice 389x389 calculee (75,466 paires, fenetre 2008-01-31 -> 2017-12-29)
Annee 2019 : matrice 425x425 calculee (90,100 paires, fenetre 2009-01-30 -> 2018-12-31)
Annee 2020 : matrice 472x472 calculee (111,156 paires, fenetre 2010-01-29 -> 2019-12-31)
Annee 2021 : matrice 512x512 calculee (130,816 paires, fenetre 2011-01-31 -> 2020-12-31)
Annee 2022 : matrice 547x547 calculee (149,331 paires, fenetre 2012-01-31 -> 2021-12-31)
Annee 2023 : matrice 573x573 calculee (163,878 paires, fenetre 2013-01-31 -> 2022-12-30)
Annee 2024 : matrice 584x584 calculee (170,236 paires, fenetre 2014-01-31 -> 2023-12-29)
Annee 2025 : matrice 570x57

### 2.2 Minimum-Variance Portfolio

In [37]:
import cvxpy as cp

# ─────────────────────────────────────────────────────────────────────────────
# PARAMÈTRE DE SHRINKAGE (fixé en dehors de la boucle)
# ─────────────────────────────────────────────────────────────────────────────
# Selon le paradoxe de Stein, l'estimateur empirique de la matrice de
# covariance est non biaisé mais très bruité lorsque N est grand par rapport
# à T=120. On "rétrécit" (shrink) cette matrice vers une cible diagonale
# très structurée (zéro corrélation supposée) afin de réduire l'erreur
# d'estimation et d'éviter que l'optimiseur de Markowitz n'exploite le bruit.
lambda_shrink = 0.1

# ─────────────────────────────────────────────────────────────────────────────
# OPTIMISATION MINIMUM-VARIANCE — une par année d'investissement (2014–2025)
# ─────────────────────────────────────────────────────────────────────────────
unconstrained_weights_shrink = {}

for year in range(2014, 2026):

    isins = filtered_carbon_isin_dict[year]
    n     = len(isins)
    Sigma = covariance_matrices[year]          # matrice empirique N×N

    # ── Scaling ×10 000 ───────────────────────────────────────────────────────
    # Multiplier la matrice de covariance par 10 000 équivaut mathématiquement
    # à exprimer les rendements en pourcentages (r% = r × 100, Cov% = Cov × 10 000).
    # Cela résout les problèmes de stabilité numérique du solveur, qui peine
    # avec des covariances de l'ordre de 10⁻⁵. Les poids optimaux finaux sont
    # identiques (le facteur d'échelle se simplifie dans le problème QP).
    Sigma_scaled = Sigma * 10_000

    # ── Shrinkage de Ledoit-Wolf (cible diagonale) ────────────────────────────
    diag         = np.diag(np.diag(Sigma_scaled))           # matrice cible : variances seules
    Sigma_shrunk = (1 - lambda_shrink) * Sigma_scaled + lambda_shrink * diag

    # ── Projection sur le cône PSD (clipping des valeurs propres) ────────────
    # La méthode Common Sample (pairwise) ne garantit pas la semi-définie
    # positivité quand N >> T. On projette sur la matrice PSD la plus proche
    # en forçant toutes les valeurs propres négatives à zéro (nearest PSD).
    eigvals, eigvecs = np.linalg.eigh(Sigma_shrunk)
    eigvals_clipped  = np.maximum(eigvals, 0.0)
    Sigma_psd_np     = eigvecs @ np.diag(eigvals_clipped) @ eigvecs.T
    Sigma_psd_np     = (Sigma_psd_np + Sigma_psd_np.T) / 2  # symétrie numérique exacte

    # ── Optimisation CVXPY / CLARABEL ─────────────────────────────────────────
    # CLARABEL est le solveur de référence pour cvxpy >= 1.4 sur les QP de
    # grande taille. Il gère nativement les matrices semi-définies positives
    # et est plus robuste qu'OSQP 1.x pour ce type de problème.
    w           = cp.Variable(n)
    objective   = cp.Minimize(cp.quad_form(w, cp.psd_wrap(Sigma_psd_np)))
    constraints = [
        cp.sum(w) == 1,   # investi à 100%
        w >= 0            # long-only
    ]
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.CLARABEL, verbose=False)

    # ── Stockage des poids ────────────────────────────────────────────────────
    if prob.status in ("optimal", "optimal_inaccurate"):
        weights = pd.Series(w.value, index=isins)
        weights = weights.clip(lower=0)             # éliminer les -ε numériques
        weights /= weights.sum()                    # renormaliser à 1
        unconstrained_weights_shrink[year] = weights
        print(f"Annee {year} : OK ({prob.status}) — {n} actifs, "
              f"poids max={weights.max():.4f}, "
              f"actifs actifs (w>0.1%)={(weights > 0.001).sum()}")
    else:
        print(f"Annee {year} : ECHEC ({prob.status}) — {n} actifs")

Annee 2014 : OK (optimal) — 254 actifs, poids max=0.2651, actifs actifs (w>0.1%)=20
Annee 2015 : OK (optimal) — 280 actifs, poids max=0.3465, actifs actifs (w>0.1%)=15
Annee 2016 : OK (optimal) — 312 actifs, poids max=0.3586, actifs actifs (w>0.1%)=19
Annee 2017 : OK (optimal) — 346 actifs, poids max=0.3817, actifs actifs (w>0.1%)=23
Annee 2018 : OK (optimal) — 389 actifs, poids max=0.3481, actifs actifs (w>0.1%)=29
Annee 2019 : OK (optimal) — 425 actifs, poids max=0.3873, actifs actifs (w>0.1%)=29
Annee 2020 : OK (optimal) — 472 actifs, poids max=0.2575, actifs actifs (w>0.1%)=26
Annee 2021 : OK (optimal) — 512 actifs, poids max=0.3460, actifs actifs (w>0.1%)=30
Annee 2022 : OK (optimal) — 547 actifs, poids max=0.2995, actifs actifs (w>0.1%)=29
Annee 2023 : OK (optimal) — 573 actifs, poids max=0.2234, actifs actifs (w>0.1%)=31
Annee 2024 : OK (optimal) — 584 actifs, poids max=0.1972, actifs actifs (w>0.1%)=29
Annee 2025 : OK (optimal) — 570 actifs, poids max=0.1723, actifs actifs (w>0

### 2.3 Comparison with the Value-Weighted Portfolio

---
# Part II — Portfolio Allocation with Carbon Emission Reduction

## 3. Allocation with a 50% Reduction in Carbon Emissions

### 3.1 Carbon Emissions

### 3.2 Long-Only Portfolio with a Carbon Footprint Objective

### 3.3 Tracking Error Minimization

### 3.4 Comparison of Portfolios

---
## 4. Portfolio Allocation with a Net Zero Objective

### 4.1 Net Zero Portfolio

### 4.2 Comparison of Portfolios